# Hall thruster 1D

`HallThruster1D` is a 1D model of a Hall thruster along its axis $z$, from the anode at
$z = 0$ through the channel to the cathode line in the near plume. Xenon enters at the anode
and drifts at a fixed speed; electrons ionize it; the ions are accelerated by the axial field,
and the electrons cross the radial magnetic field with a classical plus an anomalous collision
frequency $\nu_{ano}$ (a closure). Current continuity with the discharge voltage across the
domain sets the field and the discharge current; the electron energy equation has ohmic
heating, the ionization and excitation losses and the losses to the channel walls.

The xenon rate coefficients (Goebel and Katz's fits) and the wall loss (the exponential form
of the LANDMARK 1D case and HallThruster.jl) are PLACEHOLDERS, to be replaced by plasmacoll's
xenon data and a sheath model with secondary emission.

In [ ]:
import plasma_plots
import xarray as xr

from carafe import HallThruster1D
from carafe.hall import TwoZoneBohm

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex");

## 1. An SPT-100-like thruster

300 V and 5 mg/s of xenon through an annular channel 2.5 cm long with the SPT-100's cross
section, a radial field of 20 mT peaking at the exit, and the two-zone Bohm closure
$\nu_{ano} = c\,\omega_{ce}$ with $c = 1/160$ in the channel and $1/16$ outside
(HallThruster.jl's default). Half a millisecond on 80 cells runs past the start-up in
about ten seconds.

In [ ]:
model = HallThruster1D(
    discharge_voltage=300.0,
    mass_flow=5e-6,  # kg/s
    channel_length=0.025,
    length=0.05,
    cells=80,
)
state = model.run(5.0e-4)
data = state.to_dataset()

The performance, averaged over the second half of the run, next to the SPT-100's nominal
point (about 4.5 A, 80 mN and 1600 s):

In [ ]:
current = state.average("discharge_current")
thrust = state.average("thrust")
{
    "discharge current [A]": round(current, 2),
    "ion current [A]": round(state.average("ion_current"), 2),
    "thrust [mN]": round(1e3 * thrust, 1),
    "specific impulse [s]": round(thrust / (model.mass_flow * 9.80665)),
    "anode efficiency": round(thrust**2 / (2 * model.mass_flow * 300.0 * current), 2),
    "exit ion velocity [km/s]": round(float(state.ion_velocity[-1]) * 1e-3, 1),
}

The current is of the thruster's order, higher with this closure: fitting $\nu_{ano}$ to
the measured current is the job of the closure studies. The thrust and the specific impulse
are higher than measured: the 1D model ionizes nearly all the gas and accelerates the ions through the whole
discharge voltage along the axis: it has no beam divergence, no doubly charged ions and no
ion losses to the walls, which together take some 15–20 % of the real thruster's thrust.

## 2. The axial profiles

The neutrals are ionized in the channel, upstream of the field peak; the ions accelerate
across the exit, where the field is strongest and the electrons hottest.

In [ ]:
densities = xr.concat(
    [data.neutral_density, data.ion_density], dim="species"
).assign_coords(species=["Xe", "Xe+"])
densities.attrs.update(long_name="density", units="m^-3")
with plasma_plots.figure(5, 1, figsize=(7, 14)) as fig:
    densities.plasma.plot.profiles(x="z", over="species", ax=fig[0])
    fig[0].set_yscale("log")
    data.ion_velocity.plasma.plot.lineout(ax=fig[1])
    data.electron_temperature.plasma.plot.lineout(ax=fig[2])
    data.potential.plasma.plot.lineout(ax=fig[3])
    data.electric_field.plasma.plot.lineout(ax=fig[4])

## 3. The discharge current

The current in time, from the start: after the start-up the discharge settles into a small
oscillation, the breathing mode of the ionization front.

In [ ]:
with plasma_plots.figure(2, 1, figsize=(7, 6)) as fig:
    data.discharge_current_history.plasma.plot.lineout(ax=fig[0])
    data.thrust_history.plasma.plot.lineout(ax=fig[1])

## 4. Another closure

`anomalous` takes any callable of an `ElectronContext` (the positions, the field, the
density, the temperature, the velocities and the channel length) that returns $\nu_{ano}$
on the cells. A smaller coefficient in the channel confines the electrons better: the
current drops and the discharge breathes strongly, the oscillation the closure studies
fit against the measured spectra.

In [ ]:
confined = HallThruster1D(
    discharge_voltage=300.0,
    mass_flow=5e-6,  # kg/s
    cells=80,
    anomalous=TwoZoneBohm(inside=1 / 250, outside=1 / 16),
)
breathing = confined.run(4.0e-4, state)
trace = breathing.to_dataset().discharge_current_history
trace = trace.sel(time=slice(state.time * 1e6, None))
round(breathing.average("discharge_current", start=0.6), 2)

In [ ]:
with plasma_plots.figure(1, 1, figsize=(7, 3.6)) as fig:
    trace.plasma.plot.lineout(ax=fig[0])